# MISC

In [1]:
import os, sys
import pandas as pd
import numpy as np
from datetime import datetime
import json
import glob

notebook_dir = os.getcwd()  # Gets test folder path
project_dir = os.path.dirname(notebook_dir)  # Gets Project folder path
sys.path.append(project_dir)
print(os.getcwd())

from subproblems import *
from utils import *

d:\Jacky\Python\ADMM_P2P_Python\test


# JSON configuration

In [ ]:
# Evaluation only (Apply ML predictions to ADMM and evaluate results)
models = ['Transformer 1D-CNN', 'GRU - Vanilla', 'CNN-GRU', 'NDNN-SAM']
pred = ['pred.npy', 'pred_post.npy']
data = {
    # General info
    "config name": "config_testing_1",
    "project_name": "Eval_NDNN-SAM-T9_PrioGO-LDF-mulCES-4dec_InterGen_raw_WS_100sce",
    "device used": "AMD",
    "last_processed_index": 0,
    "number of decisions": 4,
    "sce_start": 1,
    "sce_end": 100,
    "threshold": 1e-3,
    # "threshold": -1, # for collecting convergence plots
    "bus_sys": 33,

    # Data saving info
    "is_csv": True,
    "save_csv": True,
    "injection": True,
    "injection_pros": True,
    "injection_iter": 2, # 2 = warm start
    "iter_save": 50,

    # CES configurations
    "num_ces": 2,
    "loc_ces": [19, 33],
    "cap_ces": [283.52, 275.64],

    # Prediction and dataset
    "testdataset": r"D:\Jacky\Data Output\ADMM_P2P\Database\Test and Eval\LP_PrioGO-LDF-mulCES-4dec_test_InterGen_100sce",
    "ML_model": models[3],
    "ML_model_ver": "sanT9",
    "pred_ver": pred[0], # 0 -> raw, 1 -> post-processed

    # Additional variants
    "proximal decay": False,
    "over-relaxation": False,
    "relaxation parameter": 1.5,
}

data["ADMM_ver"] = os.path.join(data["testdataset"], [f for f in os.listdir(data["testdataset"]) if f.endswith('.jl')][0])
data["primal_pred_loc"] = os.path.join(data["testdataset"], 'predictions', data["ML_model"], data["ML_model_ver"], f'primal_{data["pred_ver"]}')
data["dual_pred_loc"] = os.path.join(data["testdataset"], 'predictions', data["ML_model"], data["ML_model_ver"], f'dual_{data["pred_ver"]}')

filename = f"D:/Jacky/Python/ADMM_P2P_Python/dataset_collection/{data['config name']}.json"
with open(filename, 'w') as json_file:
    json.dump(data, json_file, indent=4) # Using indent for human-readable output

print(f"Project name            : {data['project_name']}")
print(f"ADMM version            : {os.path.basename(data['ADMM_ver'])}")
print(f"Model                   : {data['ML_model']} -> {data['ML_model_ver']}")
print(f"Prediction version      : {data['pred_ver']}")
print(f"Warm-start?             : {'Yes' if data['injection_iter']==2 else f'No, injected at iteration {data["injection_iter"]}'}")

Project name            : Eval_NDNN-SAM-T9_PrioGO-LDF-mulCES-4dec_InterGen_raw_WS_100sce
ADMM version            : PrioGO_proximal_DistFlow_AdaptiveCES_4Dec.jl
Model                   : NDNN-SAM -> sanT9
Prediction version      : pred.npy
Warm-start?             : Yes


In [ ]:
# Training and Testing dataset
gen_mode = [1, 2, 5, 10] # Odd, even, 5 steps, 10 steps
data = {
    # General info
    # "config name": "config_training_1",
    "config name": "config_InterGenTraining_1",
    "project_name": "LP_PrioGO-LDF-mulCES-4dec_test_InterGen-5steps_100sce",
    "device used": "AMD",
    "last_processed_index": 0,
    "number of decisions": 4,
    "sce_start": 1,
    "sce_end": 100,
    "threshold": 1e-3,
    # "threshold": -1,

    # Bus system info
    "bus_sys": 33,
    "gen_mode": 5,
    "gen_mode_skip": True,

    # Data saving info
    "save_csv": True,
    "iter_save": 50,

    # CES configurations
    "num_ces": 2,
    "loc_ces": [19, 33],
    "cap_ces": [283.52, 275.64],

    # Additional variants
    "ADMM_ver": r"D:\Jacky\Python\ADMM_P2P_Python\subproblems\PrioGO_proximal_DistFlow_AdaptiveCES_4Dec.jl",
    "proximal decay": False,
    "over-relaxation": False,
    "relaxation parameter": 1.5,
}
data["last_train_gen_index"] = 1 if data["gen_mode"] == 1 else 0
filename = f"D:/Jacky/Python/ADMM_P2P_Python/dataset_collection/{data['config name']}.json"
with open(filename, 'w') as json_file:
    json.dump(data, json_file, indent=4) # Using indent for human-readable output

print(f"Project name                : {data['project_name']}")
print(f"ADMM version                : {os.path.basename(data['ADMM_ver'])}")
print(f"Scenarios collected         : {data['sce_start']} ~ {data['sce_end']}")
print(f"Active Users Steps          : {data['gen_mode']}")
print(f"Number of decisions         : {data['number of decisions']}")

Project name                : LP_PrioGO-LDF-mulCES-4dec_test_InterGen-5steps_100sce
ADMM version                : PrioGO_proximal_DistFlow_AdaptiveCES_4Dec.jl
Scenarios collected         : 1 ~ 100
Active Users Steps          : 5
Number of decisions         : 4


# Pre processing

In [8]:
# General Info
folder_path = r"D:\Jacky\Data Output\ADMM_P2P\Database\Train\LP_PrioGO-LDF-mulCES-4dec_train_InterGen_1000sce"
with open(f'{folder_path}/config.json', 'r') as file:
    config = json.load(file)
total_sce = config["sce_end"] - config["sce_start"] + 1
n_sce, start_sce_save, end_sce_save = 5, config["sce_start"], config["sce_end"]
num_user = 32
hour = 48
n_bus, n_branch = 33, 32
tot_iter_save = config["iter_save"] + 1
max_timestep = 5000
n_dec = config["number of decisions"]

In [4]:
# NPZ - GRU format
primal = []
dual = []
decision_var = []

## for the sack of reshape and memory issue
primal_temp = np.load(f"{folder_path}/DecisionVariable/primal.npz")
dual_temp = np.load(f"{folder_path}/DecisionVariable/dual.npz")
decision_var_temp = np.load(f"{folder_path}/DecisionVariable/decision_var.npz")

for i in range(int(total_sce/n_sce)):
    primal.append(primal_temp[i*n_sce*num_user*n_dec*hour:(i+1)*n_sce*num_user*n_dec*hour,:].reshape(n_sce, -1, num_user, tot_iter_save, order="F"))
    dual.append(dual_temp[i*n_sce*num_user*n_dec*hour:(i+1)*n_sce*num_user*n_dec*hour,:].reshape(n_sce, -1, num_user, tot_iter_save, order="F"))
    decision_var.append(decision_var_temp[i*n_sce*num_user*8*hour:(i+1)*n_sce*num_user*8*hour,:].reshape(n_sce, -1, num_user, tot_iter_save, order="F"))
primal, dual = np.concatenate(primal, 0), np.concatenate(dual, 0)
decision_var = np.concatenate(decision_var, 0)

del primal_temp, dual_temp, decision_var_temp

In [9]:
# CSV - combine and infeasible filter
if config["save_csv"]:
    all_infeasible = []

    # 1. Read all infeasible scenario indices
    data_directory = f"{folder_path}/infeasible_sce/"
    # Search for files matching the pattern
    infeasible_files = glob.glob(os.path.join(data_directory, "infeasible_sce_*to*sce.csv"))

    if not infeasible_files:
        print("No infeasible scenario files found.")
    else:
        for file_path in infeasible_files:
            print(os.path.basename(file_path))
            df = pd.read_csv(file_path)
            infeasible_mat = df.values
            
            # Store all infeasible indices
            for inf_sce in infeasible_mat.flatten():
                all_infeasible.append(inf_sce)

        # Unique and sorted list
        all_infeasible = sorted(list(set(all_infeasible)))

        # 2. Read main CSV files
        i = start_sce_save
        while i <= end_sce_save - 4:
            print(i)
            first = i
            last = i + 4
            
            path_dv = f"{folder_path}/DecisionVariable/dual_{first}to{last}sce.csv"
            path_pv = f"{folder_path}/DecisionVariable/primal_{first}to{last}sce.csv"
            
            # Read CSVs
            df_dv = pd.read_csv(path_dv)
            df_pv = pd.read_csv(path_pv)
            
            # Convert to NumPy and Reshape
            # Note: Julia reshapes column-major (F), Python is row-major (C) by default.
            # To match Julia's reshape(5, 192, 32, 51), we use order='F'
            lambda_2dnew = df_dv.values
            lambda_4d = lambda_2dnew.reshape((5, n_dec * hour, num_user, tot_iter_save), order='F')
            
            pout_2dnew = df_pv.values
            pout_4d = pout_2dnew.reshape((5, n_dec * hour, num_user, tot_iter_save), order='F')
            
            # Concatenate along the first dimension (axis=0)
            all_dual = np.concatenate((all_dual, lambda_4d), axis=0)
            all_primal = np.concatenate((all_primal, pout_4d), axis=0)
            
            i += 5

        # 3. Adjust indices for slicing
        if start_sce_save != 1:
            # Converting to Python's 0-based indexing and adjusting by start offset
            all_infeasible = [int(x - start_sce_save) for x in all_infeasible]
        else:
            # Just shift to 0-based
            all_infeasible = [int(x - 1) for x in all_infeasible]

        # 4. Remove infeasible indices
        # Create a mask of indices to keep
        total_scenarios = all_dual.shape[0]
        remaining_indices = [idx for idx in range(total_scenarios) if idx not in all_infeasible]

        new_all_dual = all_dual[remaining_indices, :, :, :]
        new_all_primal = all_primal[remaining_indices, :, :, :]

        # 5. Reshape back to 2D and Save
        # To match Julia's CSV output, we use order='F' during the flatten
        lambda_2d = new_all_dual.reshape(-1, all_dual.shape[3], order='F')
        pout_2d = new_all_primal.reshape(-1, all_primal.shape[3], order='F')

        dv = pd.DataFrame(lambda_2d)
        pv = pd.DataFrame(pout_2d)

        out_path_dual = f"{folder_path}/training ready/dual_{start_sce_save}to{end_sce_save}sce_feasible.csv"
        out_path_primal = f"{folder_path}/training ready/primal_{start_sce_save}to{end_sce_save}sce_feasible.csv"

        dv.to_csv(out_path_dual, index=False)
        pv.to_csv(out_path_primal, index=False)

        print(f"Total feasible scenario = {total_sce} - {len(all_infeasible)} = {total_sce - len(all_infeasible)}")

No infeasible scenario files found.


In [5]:
# NPZ - infeasible filter
if not config["save_csv"]:
    all_dual = None
    all_primal = None
    all_infeasible = []

    # 1. Read all infeasible scenario indices
    data_directory = f"{folder_path}/infeasible_sce/"
    # Search for files matching the pattern
    infeasible_files = glob.glob(os.path.join(data_directory, "infeasible_sce_*to*sce.csv"))

    if not infeasible_files:
        print("No infeasible scenario files found.")
    else:
        for file_path in infeasible_files:
            print(os.path.basename(file_path))
            df = pd.read_csv(file_path)
            infeasible_mat = df.values
            
            # Store all infeasible indices
            for inf_sce in infeasible_mat.flatten():
                all_infeasible.append(inf_sce)

        # Unique and sorted list
        all_infeasible = sorted(list(set(all_infeasible)))
        
        # 3. Adjust indices for slicing
        if start_sce_save != 1:
            # Converting to Python's 0-based indexing and adjusting by start offset
            all_infeasible = [int(x - start_sce_save) for x in all_infeasible]
        else:
            # Just shift to 0-based
            all_infeasible = [int(x - 1) for x in all_infeasible]

        # 4. Remove infeasible indices
        # Create a mask of indices to keep
        total_scenarios = dual.shape[0]
        remaining_indices = [idx for idx in range(total_scenarios) if idx not in all_infeasible]

        dual = dual[remaining_indices, :, :, :]
        primal = primal[remaining_indices, :, :, :]
        decision_var = decision_var[remaining_indices, :, :, :]

        print(f"Total feasible scenario = {total_sce} - {len(all_infeasible)} = {total_sce - len(all_infeasible)}")

No infeasible scenario files found.


In [6]:
# Save the filtered data
os.makedirs(f"{folder_path}/training ready", exist_ok=True)
np.save(f"{folder_path}/training ready/primalGRU", primal)
np.save(f"{folder_path}/training ready/dualGRU", dual)
np.save(f"{folder_path}/training ready/decisionGRU", decision_var)